In [33]:
import os
from huggingface_hub import InferenceClient
from pathlib import Path
from dotenv import load_dotenv

# Load BASE configuration
PDF_DIR = Path.cwd().parent.parent / "Database" / "PDF"
PDF_SALES_GRT_1 = PDF_DIR / "sales-growth-1.pdf"
PDF_SALES_GRT_2 = PDF_DIR / "sales-growth-2.pdf"
PDF_SALES_GRT_3 = PDF_DIR / "sales-growth-3-count.pdf"

ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")

# LLM flows structure
* #### 1. Document sources
* #### 2. Document processing
* #### 3. Text Chunking
* #### 4. Embedding Generation
* #### 5. Vector Database Index
* ### User Query → Query Embedding → Similarity Search
* ### 6. Retrieved Context Chunks
* ### 7. LLM Generation
* ### 8. Final Response

## Use Document sources to increase LLM vectors engagement while not must be the same at Documents. But, can help the answer clearly on Database

## 1. Document Sources

In [3]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader

loader = PyPDFLoader(PDF_SALES_GRT_1)
documents = loader.load()

## 2. Document Processing Chunking

In [ ]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)
print(f"After splitting: {len(chunks)} chunks created.")

After splitting: 95 chunks created.


## 3. Embedding generation

In [ ]:
from langchain_classic.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/s0/01h8dy1902zgb59dr1yqy6jr0000gn/T/ipykernel_18410/2297378262.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 4, VectorStore Database Index

In [ ]:
from langchain_classic.vectorstores import FAISS

vector_db = FAISS.from_documents(chunks, embeddings)

## 5. Retrieved Context Chunks

In [ ]:
from langchain_classic.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Pipeline for open-source LLMs models hosted on HuggingFace Hub
llm_endpoint = HuggingFaceEndpoint(
    repo_id="microsoft/Phi-3-mini-4k-instruct",
    task="text-generation",
    max_new_tokens=256,
    temperature=0.2,
    huggingfacehub_api_token=HUGGINGFACE_API_KEY
)

# Convert to a chat model
llm = ChatHuggingFace(llm=llm_endpoint)

# Setup QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db.as_retriever()
)


## 6. LLM response on Query LLMs

In [ ]:
import re

response = qa_chain.run("What is strategy to improve forecasting and decision making in sales growth?")
sentences = re.split(r'(?<=[.!?])\s+', response.strip())

for sentence in sentences:
    if sentence:
        print(sentence)

Based on the provided text, the strategy to improve forecasting and decision-making in sales growth involves **leveraging AI to analyze historical data, customer behavior, market trends, and external factors** to generate more accurate sales forecasts.
This enables data-driven decisions in areas such as inventory management, resource allocation, and budget planning.
Key elements of this strategy include:
- **AI-powered forecasting**: Moving beyond traditional, inexact methods to use AI for precise predictions.
- **Improved resource allocation**: Better forecasting allows for smarter distribution of inventory and human resources.
- **Prioritization of high-value deals**: Clearer pipeline visibility helps focus efforts on the most promising opportunities.
- **Automation of routine tasks**: Frees sales teams to focus on higher-value activities, boosting productivity.
The overall impact is a more agile, informed sales function capable of consistently meeting targets and adapting to market 